# Experiment: overlapping-tile Schwarz for stuck fold slices

**Hypothesis.** The slices the main cluster runner (`_run_2d_clusters.py`) fails on all
fail the same way: a dense fold core that `MERGE_DILATION` fuses into one giant connected
component, producing a single SLSQP crop too large to solve (times out / OOM-crashes the
worker, or exceeds `MAX_CLUSTER_CELLS`).

This notebook tests an alternative: instead of one monolithic crop, tile the fold region
into **small overlapping tiles** and solve them as an **overlapping Schwarz domain
decomposition** — multiplicative (Gauss-Seidel) sweeps, each tile solved with frozen edges
at the *current* field, the overlap providing the coupling between tiles.

- Each tile solve reuses the **exact same solver** the runner uses
  (`_bench_worker.solve_cluster_inline`): multi-pass L2 SLSQP + L1 polish, analytical
  Jacobian, frozen-edge boundary.
- Tiles overlap by `overlap` cells; a corner inside the overlap is *movable* in more than
  one tile, so corrections propagate between tiles across sweeps.
- Multiplicative / Gauss-Seidel: tiles are spliced back immediately, so the next tile in
  the sweep sees the updated field. Sweep direction alternates (symmetric Gauss-Seidel).

Nothing in `_run_2d_clusters.py` / `_bench_worker.py` is modified — this notebook only
imports and calls them. It is an experiment, run serially (no subprocess pool), so the
per-tile timing is visible.

**Stuck slices tested:** the 7 slices marked `feasible=False` by the main run
(z = 30, 54, 57, 58, 61, 150, 290).

In [ ]:
import os, sys, time

# Locate the repo root (walk up until we see both data/ and notebooks/).
_d = os.path.abspath('.')
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'data')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        REPO = _d
        break
    _d = os.path.dirname(_d)
else:
    raise RuntimeError('could not locate repo root')
MANUSCRIPT = os.path.join(REPO, 'notebooks', 'manuscript')
sys.path.insert(0, REPO)
sys.path.insert(0, MANUSCRIPT)

import numpy as np
import matplotlib.pyplot as plt

from dvfopt.jacobian.triangle_sign import _triangle_areas_2d
from _bench_worker import solve_cluster_inline

THRESHOLD = 0.01      # 2-triangle constraint lower bound
EPS_L1 = 1e-4         # smoothed-L1 epsilon
DATA_PATH = os.path.join(REPO, 'data', 'corrected_correspondences_count_touching',
                         'registered_output', 'deformation3d.npy')
STUCK_Z = [30, 54, 57, 58, 61, 150, 290]
print(f'repo       : {REPO}')
print(f'data       : {DATA_PATH}')
print(f'stuck z    : {STUCK_Z}')

In [ ]:
phi_full = np.load(DATA_PATH)   # (3, D, H, W) with [dz, dy, dx]
D, H, W = phi_full.shape[1:]
print(f'deformation3d.npy: {phi_full.shape}  (D={D}, H={H}, W={W})')


def fold_stats(phi):
    """phi: (2, H, W) stack of [dy, dx]. Returns (n_neg_tri, min_tri)."""
    T1, T2 = _triangle_areas_2d(phi[0], phi[1])
    n_neg = int((T1 <= 0).sum() + (T2 <= 0).sum())
    return n_neg, float(min(T1.min(), T2.min()))


def slice_phi(z):
    """The (2, H, W) [dy, dx] field for slice z, from the original DVF."""
    return np.stack([phi_full[1, z].copy(), phi_full[2, z].copy()])


print(f'\n{"z":>5s}  {"init_n_neg_tri":>14s}  {"init_min_tri":>12s}')
for z in STUCK_Z:
    n, m = fold_stats(slice_phi(z))
    print(f'{z:5d}  {n:14d}  {m:+12.4f}')

## Method

`solve_overlapping_schwarz` runs multiplicative Schwarz sweeps:

1. Each sweep, find the bounding box of all currently-folded cells.
2. Tile that bbox with `tile`-cell tiles, stepping by `tile - overlap` so adjacent tiles
   share an `overlap`-cell band.
3. For each tile that still contains a fold: build a rectangular `interior_mask`
   (frozen 1-corner ring), extract the crop from the **current** field, and solve it with
   `solve_cluster_inline`. Splice the tile's interior corners back **immediately** so the
   next tile in the sweep sees the update (Gauss-Seidel).
4. Sweep direction alternates each sweep (symmetric Gauss-Seidel).
5. Stop when `n_neg == 0` or `max_sweeps` is reached.

The overlap is what carries a correction from one tile into its neighbour: a corner in the
shared band is movable in both tiles, and frozen-at-current-value when the other tile is
solved — so no averaging is needed and no tile can silently re-break its neighbour.

In [ ]:
def make_tiles(bbox, H, W, tile, overlap):
    """Overlapping tiles (cell coords) covering bbox=(cy0,cy1,cx0,cx1).
    Each tile is (y0, y1, x0, x1) with y1<=H-1, x1<=W-1 (corner crop is
    [y0:y1+1, x0:x1+1])."""
    cy0, cy1, cx0, cx1 = bbox
    stride = max(1, tile - overlap)
    out = set()
    y_starts = list(range(cy0, max(cy0 + 1, cy1), stride))
    x_starts = list(range(cx0, max(cx0 + 1, cx1), stride))
    for y0 in y_starts:
        for x0 in x_starts:
            y1 = min(y0 + tile, H - 1)
            x1 = min(x0 + tile, W - 1)
            y0c = max(0, y1 - tile)
            x0c = max(0, x1 - tile)
            if (y1 - y0c) >= 4 and (x1 - x0c) >= 4:
                out.add((y0c, y1, x0c, x1))
    return sorted(out)


def solve_overlapping_schwarz(phi0, phi_anchor, *, tile=32, overlap=8,
                              max_sweeps=15, l2_max_passes=10,
                              l2_max_iter=80, l1_max_iter=100,
                              verbose=True):
    """Overlapping-tile multiplicative Schwarz fold correction.

    phi0, phi_anchor : (2, H, W) current field and the original anchor.
    Returns (phi_corrected, history) where history is a list of per-sweep
    dicts {sweep, n_neg, min_tri, n_tiles, t}.
    """
    H_, W_ = phi0.shape[1], phi0.shape[2]
    phi = phi0.copy()
    history = []
    n, m = fold_stats(phi)
    history.append(dict(sweep=0, n_neg=n, min_tri=m, n_tiles=0, t=0.0))
    if verbose:
        print(f'  init      : n_neg={n:5d}  min_tri={m:+.4f}')
    for sweep in range(1, max_sweeps + 1):
        T1, T2 = _triangle_areas_2d(phi[0], phi[1])
        fold = np.minimum(T1, T2) <= 0
        if not fold.any():
            break
        ys, xs = np.where(fold)
        bbox = (int(ys.min()), int(ys.max()) + 1,
                int(xs.min()), int(xs.max()) + 1)
        tiles = make_tiles(bbox, H_, W_, tile, overlap)
        if sweep % 2 == 0:
            tiles = tiles[::-1]        # alternate sweep direction
        t0 = time.time()
        n_solved = 0
        for (y0, y1, x0, x1) in tiles:
            phi_win = phi[:, y0:y1 + 1, x0:x1 + 1].copy()
            t1w, t2w = _triangle_areas_2d(phi_win[0], phi_win[1])
            if not (np.minimum(t1w, t2w) <= 0).any():
                continue              # tile has no fold right now
            sy, sx = y1 - y0, x1 - x0
            im = np.zeros((sy + 1, sx + 1), dtype=bool)
            im[1:-1, 1:-1] = True
            c = dict(cluster_id=0, z=-1, y0=y0, y1=y1, x0=x0, x1=x1,
                     crop_cells_y=sy, crop_cells_x=sx,
                     component_cells=int((np.minimum(t1w, t2w) <= 0).sum()),
                     interior_mask=im, skipped_too_large=False)
            anc_win = phi_anchor[:, y0:y1 + 1, x0:x1 + 1].copy()
            _row, phi_l1 = solve_cluster_inline(
                c, phi_win, anc_win, THRESHOLD, EPS_L1,
                l2_max_passes, l2_max_iter, l1_max_iter)
            if phi_l1 is not None:
                yy, xx = np.where(im)
                phi[:, y0 + yy, x0 + xx] = phi_l1[:, yy, xx]
            n_solved += 1
        n, m = fold_stats(phi)
        dt = time.time() - t0
        history.append(dict(sweep=sweep, n_neg=n, min_tri=m,
                             n_tiles=n_solved, t=dt))
        if verbose:
            print(f'  sweep {sweep:2d}  : n_neg={n:5d}  min_tri={m:+.4f}  '
                  f'tiles_solved={n_solved:4d}  ({dt:.0f}s)')
        if n == 0:
            break
    return phi, history

## Single-slice demo

Run the overlapping-tile Schwarz solver on one stuck slice (z = 30, the lightest of the
stuck set) so the per-sweep behaviour and timing are visible.

In [ ]:
DEMO_Z = 30
TILE = 32        # tile size in cells
OVERLAP = 8      # overlap band in cells (stride = TILE - OVERLAP)

phi_init = slice_phi(DEMO_Z)
print(f'z={DEMO_Z}  (tile={TILE}, overlap={OVERLAP})')
t0 = time.time()
phi_demo, hist_demo = solve_overlapping_schwarz(
    phi_init, phi_init, tile=TILE, overlap=OVERLAP, max_sweeps=15)
print(f'\nwall: {time.time()-t0:.0f}s')
n_fin, m_fin = fold_stats(phi_demo)
print(f'final: n_neg={n_fin}  min_tri={m_fin:+.4f}  '
      f'-> {"CONVERGED" if n_fin == 0 else "still folded"}')

In [ ]:
# Before/after min(T1,T2) maps + the per-sweep fold-count curve.
T1i, T2i = _triangle_areas_2d(phi_init[0], phi_init[1])
T1f, T2f = _triangle_areas_2d(phi_demo[0], phi_demo[1])
init_min = np.minimum(T1i, T2i)
fin_min = np.minimum(T1f, T2f)
vmax = max(abs(init_min.min()), abs(fin_min.min()), 1.0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)
im0 = axes[0].imshow(init_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title(f'z={DEMO_Z} initial  min(T1,T2)\nfolded cells red')
axes[1].imshow(fin_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('after overlapping-tile Schwarz')
for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im0, ax=axes[:2], shrink=0.8, label='min(T1, T2)')

sw = [h['sweep'] for h in hist_demo]
nn = [h['n_neg'] for h in hist_demo]
axes[2].plot(sw, nn, 'o-', color='#c62828')
axes[2].set_yscale('symlog', linthresh=1)
axes[2].set_xlabel('Schwarz sweep'); axes[2].set_ylabel('n_neg_tri (symlog)')
axes[2].set_title('folded triangles vs sweep')
axes[2].grid(alpha=0.3)
plt.show()

## All stuck slices

Run the same solver on every stuck slice. This is serial (one tile solve at a time), so it
takes a while — the per-slice wall time is part of "how it runs". Lower `max_sweeps` or
shorten `STUCK_Z` to a subset for a quicker pass.

In [ ]:
results = []
for z in STUCK_Z:
    phi_z = slice_phi(z)
    n0, m0 = fold_stats(phi_z)
    print(f'=== z={z}  (init n_neg={n0}) ===')
    t0 = time.time()
    phi_c, hist = solve_overlapping_schwarz(
        phi_z, phi_z, tile=TILE, overlap=OVERLAP, max_sweeps=15)
    wall = time.time() - t0
    nf, mf = fold_stats(phi_c)
    results.append(dict(z=z, init_n_neg=n0, final_n_neg=nf, final_min_tri=mf,
                         sweeps=hist[-1]['sweep'], wall=wall,
                         converged=(nf == 0)))
    print(f'    -> final n_neg={nf}  min_tri={mf:+.4f}  '
          f'sweeps={hist[-1]["sweep"]}  wall={wall:.0f}s\n')

print(f'{"z":>5s} {"init":>6s} {"final":>6s} {"min_tri":>9s} '
      f'{"sweeps":>7s} {"wall_s":>8s}  result')
for r in results:
    print(f'{r["z"]:5d} {r["init_n_neg"]:6d} {r["final_n_neg"]:6d} '
          f'{r["final_min_tri"]:+9.4f} {r["sweeps"]:7d} {r["wall"]:8.0f}  '
          f'{"CONVERGED" if r["converged"] else "still folded"}')
n_ok = sum(r['converged'] for r in results)
print(f'\nconverged: {n_ok}/{len(results)} stuck slices')

## Notes / interpretation

_(Fill in after running.)_

Things to read off the results:

- **Did it converge?** `converged` column — if overlapping-tile Schwarz reaches `n_neg=0`
  on slices the monolithic connected-component solver failed on, the tiling idea works.
- **Sweep curve.** A steadily-decreasing `n_neg` per sweep means the overlap is
  successfully propagating corrections. A curve that plateaus above 0 means some folds are
  stuck on tile seams or need a wider coordinated motion than one tile allows — try a
  larger `TILE`, or more `OVERLAP`.
- **Cost.** Wall time here is serial; a real implementation would solve non-overlapping
  tiles within a sweep in parallel (checkerboard colouring), as the main runner does for
  connected-component clusters.
- **Knobs.** `TILE` (displacement budget per tile vs SLSQP problem size) and `OVERLAP`
  (coupling strength between tiles) are the two to sweep if convergence stalls.